In [2]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CLEAN_PATH = Path("../data/processed/netflix_clean.csv")

df = pd.read_csv(CLEAN_PATH, parse_dates=["date_added"])

# Titles are not fully unique, so drop repeats to keep one row per title
df = df.drop_duplicates(subset="title").reset_index(drop=True)
print(f"Titles for the recommender: {len(df)}")

df[["title", "type", "director", "primary_country", "listed_in"]].head()

Titles for the recommender: 8786


,title,type,director,primary_country,listed_in
0,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,Documentaries
1,Ganglands,TV Show,Julien Leclercq,France,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,TV Show,Mike Flanagan,United States,"TV Dramas, TV Horror, TV Mysteries"
3,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,"Children & Family Movies, Comedies"
4,Sankofa,Movie,Haile Gerima,United States,"Dramas, Independent Movies, International Movies"


In [4]:
def build_soup(row: pd.Series) -> str:
    """Combine genres, director and country into one lowercase text string."""
    genres = [g.strip().replace(" ", "") for g in row["listed_in"].split(",")]
    directors = [] if row["director"] == "Unknown" else [
        d.strip().replace(" ", "") for d in row["director"].split(",")
    ]
    country = [] if row["primary_country"] == "Unknown" else [
        row["primary_country"].replace(" ", "")
    ]
    # Genres are repeated so they weigh more than director and country
    tokens = genres * 3 + directors + country + [row["type"].replace(" ", "")]
    return " ".join(tokens).lower()


df["soup"] = df.apply(build_soup, axis=1)
df[["title", "soup"]].head()

,title,soup
0,Dick Johnson Is Dead,documentaries documentaries documentaries kirs...
1,Ganglands,crimetvshows internationaltvshows tvaction&adv...
2,Midnight Mass,tvdramas tvhorror tvmysteries tvdramas tvhorro...
3,Confessions of an Invisible Girl,children&familymovies comedies children&family...
4,Sankofa,dramas independentmovies internationalmovies d...


In [5]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["soup"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

# Look-up table: title -> row position
indices = pd.Series(df.index, index=df["title"])

TF-IDF matrix shape: (8786, 5390)


In [6]:
def recommend(title: str, n: int = 5) -> pd.DataFrame:
    """Return the n titles most similar to the given title."""
    if title not in indices:
        raise KeyError(f"'{title}' not found in the dataset")

    idx = indices[title]
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
    top = scores.argsort()[::-1][1 : n + 1]  # skip the title itself

    result = df.iloc[top][["title", "type", "listed_in", "primary_country"]].copy()
    result["similarity"] = scores[top].round(3)
    return result


recommend("Ganglands")

,title,type,listed_in,primary_country,similarity
8404,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",France,0.938
6734,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",France,0.938
8330,Agent Raghav,TV Show,"Crime TV Shows, International TV Shows, TV Act...",India,0.910
486,The Truth,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Pakistan,0.905
575,Fatal Destiny,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Pakistan,0.905


In [7]:
import numpy as np

genre_sets = df["listed_in"].apply(lambda s: {g.strip() for g in s.split(",")})


def genre_jaccard(a: int, b: int) -> float:
    return len(genre_sets[a] & genre_sets[b]) / len(genre_sets[a] | genre_sets[b])


def evaluate(sample_size: int = 200, n: int = 5, seed: int = 42) -> dict:
    """Average genre overlap and same-type rate for random sample titles."""
    rng = np.random.default_rng(seed)
    sample = rng.choice(len(df), size=sample_size, replace=False)

    jaccards, same_type = [], []
    for idx in sample:
        scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
        top = scores.argsort()[::-1][1 : n + 1]
        jaccards.extend(genre_jaccard(idx, j) for j in top)
        same_type.extend(df.loc[j, "type"] == df.loc[idx, "type"] for j in top)

    return {
        "titles evaluated": sample_size,
        "avg genre overlap (Jaccard)": round(float(np.mean(jaccards)), 3),
        "same type rate": round(float(np.mean(same_type)), 3),
    }


evaluate()

{'titles evaluated': 200,
 'avg genre overlap (Jaccard)': 0.959,
 'same type rate': 1.0}

In [8]:
for title in ["Sankofa", "Midnight Mass"]:
    print(f"Because you watched: {title}")
    display(recommend(title))

Because you watched: Sankofa


,title,type,listed_in,primary_country,similarity
5863,Shanghai,Movie,"Dramas, Independent Movies, International Movies",India,0.705
280,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",India,0.705
5160,Iqbal,Movie,"Dramas, Independent Movies, International Movies",India,0.699
6375,Traffic Signal,Movie,"Dramas, Independent Movies, International Movies",India,0.699
4675,Chittagong,Movie,"Dramas, Independent Movies, International Movies",United States,0.697


Because you watched: Midnight Mass


,title,type,listed_in,primary_country,similarity
7906,The Haunting of Hill House,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.951
7958,The Originals,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.951
7072,Ratched,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.951
6615,Brand New Cherry Flavor,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.951
7042,The Haunting of Bly Manor,TV Show,"TV Dramas, TV Horror, TV Mysteries",United States,0.951


## Summary

**Approach:** content-based recommender. For each title, genres (repeated 3x for weight), director, country and type were combined into one text "soup", converted with TF-IDF, and compared using cosine similarity.

**Evaluation (200 random titles, 5 recommendations each):**

| Metric | Result |
|---|---|
| Average genre overlap (Jaccard) | 0.959 |
| Recommendations of the same type (Movie / TV Show) | 100% |

**Examples:** "Ganglands" returns French and international crime shows (Lupin, Crime Time); "Midnight Mass" returns US horror and mystery series (The Haunting of Hill House, Ratched).

**Limitations:** the dataset has no description, cast or user ratings, so recommendations rely mostly on genres. Because genres are weighted heavily, the overlap score is naturally high. Adding plot descriptions would improve quality.